# DDPM Test on MNIST Dataset

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from jax.lib import xla_bridge
import flax.linen as nn
from flax.training import train_state
import optax
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt



In [ ]:
T = 1000
batch_size = 128 # Increased batch size for more stable training
image_shape = (28, 28, 1) # MNIST image shape

# define hyper-parameters 
beta_start = 0.0001
beta_end = 0.02
betas = beta_start * (beta_end / beta_start) ** (jnp.linspace(0, 1, T))
alphas = 1 - betas
alpha_bars = jnp.cumprod(alphas)

In [ ]:
class UNet(nn.Module):
    T: int
    
    def setup(self):
        # Time embedding
        self.time_embed_mlp = nn.Sequential([
            nn.Dense(64), nn.relu, nn.Dense(64)
        ])

        self.time_dense_h2 = nn.Dense(32) 
        
        # --- Down Path ---
        # 28x28
        self.conv1 = nn.Conv(features=16, kernel_size=(3, 3), padding='SAME')
        # 14x14
        self.conv2 = nn.Conv(features=32, kernel_size=(3, 3), padding='SAME', strides=(2, 2))
        # 7x7
        self.conv3 = nn.Conv(features=64, kernel_size=(3, 3), padding='SAME', strides=(2, 2))

        # --- Bottleneck ---
        self.bottleneck = nn.Conv(features=128, kernel_size=(3, 3), padding='SAME')

        # --- Up Path ---
        # 14x14
        self.upconv1 = nn.ConvTranspose(features=64, kernel_size=(3, 3), strides=(2, 2), padding='SAME')
        self.conv4 = nn.Conv(features=64, kernel_size=(3, 3), padding='SAME')
        # 28x28
        self.upconv2 = nn.ConvTranspose(features=32, kernel_size=(3, 3), strides=(2, 2), padding='SAME')
        self.conv5 = nn.Conv(features=16, kernel_size=(3, 3), padding='SAME')
        
        # --- Final Conv ---
        self.final_conv = nn.Conv(features=1, kernel_size=(1, 1)) # Output 1 channel (the noise)

    def __call__(self, x, t):
        # x shape: [B, 28, 28, 1]
        # t shape: [B,]

        # Sinusoidal time embedding
        half_dim = 32
        emb = jnp.log(10000) / (half_dim - 1)
        emb = jnp.exp(jnp.arange(half_dim) * -emb)
        emb = t[:, None] * emb[None, :]
        emb = jnp.concatenate([jnp.sin(emb), jnp.cos(emb)], axis=-1) # [B, 64]
        t_embed = self.time_embed_mlp(emb) # [B, 64]

        # --- Down Path ---
        h1 = nn.relu(self.conv1(x)) # [B, 28, 28, 16]
        
        h2 = nn.relu(self.conv2(h1)) # [B, 14, 14, 32]
        # Add time embedding
        t_embed_h2 = self.time_dense_h2(t_embed)
        h2 = h2 + t_embed_h2[:, None, None, :] # Broadcast add
        
        h3 = nn.relu(self.conv3(h2)) # [B, 7, 7, 64]

        # --- Bottleneck ---
        bn = nn.relu(self.bottleneck(h3)) # [B, 7, 7, 128]

        # --- Up Path ---
        up1 = self.upconv1(bn) # [B, 14, 14, 64]
        up1 = jnp.concatenate([up1, h2], axis=-1) # Skip connection
        up1 = nn.relu(self.conv4(up1)) # [B, 14, 14, 64]

        up2 = self.upconv2(up1) # [B, 28, 28, 32]
        up2 = jnp.concatenate([up2, h1], axis=-1) # Skip connection
        up2 = nn.relu(self.conv5(up2)) # [B, 28, 28, 16]
        
        # --- Final ---
        out = self.final_conv(up2) # [B, 28, 28, 1]
        return out

In [ ]:
def load_dataset(batch_size):
    tf.config.experimental.set_visible_devices([], "GPU")
    
    def preprocess(data):
        # MNIST is [0, 255], uint8. We need [-1, 1], float32.
        img = tf.cast(data['image'], tf.float32)
        img = img / 127.5 - 1.0 # Normalize to [-1, 1]
        return img
    
    ds = tfds.load('mnist', split='train', as_supervised=False)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).cache()
    ds = ds.shuffle(60000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # Convert to numpy for JAX
    return tfds.as_numpy(ds)

In [ ]:
def create_train_state(model, key):
    # Dummy inputs for initialization
    t = jnp.zeros((batch_size,), dtype=jnp.int32)
    dummy_x = jnp.ones((batch_size, *image_shape))
    
    params = model.init(key, dummy_x, t)['params']
    tx = optax.chain(
        optax.clip(1.0),
        optax.adam(1e-4) 
    )
    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

def compute_loss(params, apply_fn, key, x0):
    batch_size = x0.shape[0]
    key1, key2 = jr.split(key, 2)
    
    noise = jr.normal(key1, x0.shape)

    t = jr.randint(key2, (batch_size,), 0, T)

    alpha_t = alpha_bars[t].reshape(-1, 1, 1, 1) 
    
    xt = jnp.sqrt(alpha_t) * x0 + jnp.sqrt(1 - alpha_t) * noise
    pred_noise = apply_fn({"params": params}, xt, t)

    return jnp.mean((pred_noise - noise)**2)

@jax.jit
def train_step(state, batch, key):
    loss, grad = jax.value_and_grad(compute_loss)(state.params, state.apply_fn, key, batch)
    return state.apply_gradients(grads=grad), loss


In [ ]:
def _sample_fn(apply_fn, params, key, num_samples=64):
    """
    Runs the full sampling loop using lax.scan by capturing apply_fn/params.
    """

    def _sample_scan_body(carry, t):

        samples, key = carry

        num_samples_local = samples.shape[0]
        t_arr = jnp.full((num_samples_local,), t)
        pred_noise = apply_fn({'params': params}, samples, t_arr) 

        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]
        beta_t = betas[t]

        samples_t_minus_1 = (samples - 
                           (beta_t / jnp.sqrt(1 - alpha_bar_t)) * pred_noise) / jnp.sqrt(alpha_t)

        key, key_ = jr.split(key)

        def add_noise(s):
            z = jr.normal(key_, (num_samples_local, *image_shape))
            return s + jnp.sqrt(beta_t) * z

        def no_noise(s):
            return s

        samples_t_minus_1 = jax.lax.cond(t > 0, add_noise, no_noise, samples_t_minus_1)

        new_carry = (samples_t_minus_1, key)
        output = samples # Store x_t
        
        return new_carry, output

    key, key_ = jr.split(key)
    init_samples = jr.normal(key_, (num_samples, *image_shape))

    init_carry = (init_samples, key)

    timesteps = jnp.arange(T - 1, -1, -1)
    final_carry, all_outputs = jax.lax.scan(_sample_scan_body, init_carry, timesteps)
    
    x_0 = final_carry[0]
    
    all_samples = jnp.concatenate([all_outputs, x_0[None, ...]], axis=0)
    
    return all_samples


sample = jax.jit(_sample_fn, static_argnums=(0, 3))

In [ ]:
def train(model, key, init_state, dataset_iter, num_epochs=20):
    state = init_state
    losslist = []

    print("Starting training...")
    for epoch in range(num_epochs):
        epoch_loss = 0.
        num_batches = 0
        for batch in dataset_iter:
            key, key_ = jr.split(key)
            state, loss = train_step(state, batch, key_)
            epoch_loss += loss
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches
        losslist.append(avg_loss)
        print(f'Epoch {epoch}, Avg. Loss: {avg_loss}')
    
    # plot loss
    plt.figure(figsize=(10, 4))
    plt.plot(losslist, label='Loss', color='blue', alpha=0.7)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
    
    return model, state

In [ ]:
def denormalize(img):
    # Denormalize from [-1, 1] to [0, 1]
    return (img + 1) / 2.0

def plot_forward_process(images, num_images=5):
    """Plots the forward (noising) process."""
    print("Plotting forward noising process...")
    x0 = images[:num_images] 
    key = jr.key(42)
    noise = jr.normal(key, x0.shape)
    
    timesteps = jnp.array([0, T//4, T//2, 3*T//4, T-1])
    
    fig, axes = plt.subplots(num_images, len(timesteps) + 1, figsize=(10, num_images))
    
    for i in range(num_images):
        axes[i, 0].imshow(denormalize(x0[i]).squeeze(), cmap='gray')
        axes[i, 0].set_title("t=0")
        axes[i, 0].axis('off')
        
        for j, t in enumerate(timesteps):
            alpha_t = alpha_bars[t]
            xt = jnp.sqrt(alpha_t) * x0[i] + jnp.sqrt(1 - alpha_t) * noise[i]
            
            axes[i, j+1].imshow(denormalize(xt).squeeze(), cmap='gray')
            axes[i, j+1].set_title(f"t={t}")
            axes[i, j+1].axis('off')
            
    plt.tight_layout()
    plt.show()

def plot_backward_process(samples_list, num_images_to_plot=5):
    """Plots the backward (denoising) process."""
    print("Plotting backward denoising process...")

    num_steps = len(samples_list)
    indices_to_plot = jnp.linspace(0, num_steps - 1, 5, dtype=jnp.int32)
    
    fig, axes = plt.subplots(num_images_to_plot, len(indices_to_plot), figsize=(10, num_images_to_plot))
    
    for i in range(num_images_to_plot):
        for j, list_idx in enumerate(indices_to_plot):
            img_batch = samples_list[list_idx]
            img = denormalize(img_batch[i])
            img = jnp.clip(img, 0.0, 1.0)
            
            axes[i, j].imshow(img.squeeze(), cmap='gray')
            axes[i, j].axis('off')
            if i == 0:
                # Calculate approximate 't' value
                t = T - (list_idx * (T / (num_steps - 1)))
                axes[i, j].set_title(f"t≈{t:.0f}")
                
    plt.tight_layout()
    plt.show()

In [ ]:
print(f"JAX is running on: {xla_bridge.get_backend().platform}")

# Load data
train_loader = load_dataset(batch_size)

# Get a batch for plotting the forward process
first_batch = next(iter(train_loader))

# Initialize model and state
model = UNet(T)
key = jr.key(0)
key, key_ = jr.split(key)
state = create_train_state(model, key_)

# Train
key, key_ = jr.split(key)
_, final_state = train(model, key_, state, train_loader, num_epochs=200)

In [ ]:
plot_forward_process(first_batch, num_images=5)

In [ ]:
print("Sampling from the model...")
key, key_ = jr.split(key)
samples_list = sample(final_state.apply_fn, final_state.params, key_, num_samples=10)
plot_backward_process(samples_list, num_images_to_plot=5)

### Generate 3*3 Grid Results

In [25]:
import numpy as np
import os

In [ ]:
def create_image_grid(images, num_rows, num_cols, filename):
    """
    Creates a single grid image from a batch of images and saves it.
    images: JAX/Numpy array of shape (N, H, W, 1) where N=num_rows*num_cols.
    """
    # Detach from JAX, convert to numpy, and normalize/clip for plotting
    images = np.array(images)
    images = np.clip(images, 0, 1)
    
    # Reshape the images into a grid
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 1.5, num_rows * 1.5))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < len(images):
            # Remove the channel dimension (H, W, 1) -> (H, W)
            ax.imshow(images[i, :, :, 0], cmap='gray')
            ax.axis('off')
        else:
            ax.axis('off') 

    plt.subplots_adjust(wspace=0.01, hspace=0.01) # Minimize margins
    
    # Ensure the output directory exists
    output_dir = 'DDPM_result'
    os.makedirs(output_dir, exist_ok=True)
    
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    print(f"Saved: {save_path}")

In [ ]:
def get_forward_chain(data_loader, timesteps, key, batch_size=9):
    """
    Generates the forward diffusion chain starting from a batch of real images.
    """
    
    x_start = data_loader[:batch_size]
    
    chain_images = []
    
    for t in timesteps:
        t_arr = jnp.full((batch_size,), t)
        
        key, key_ = jr.split(key)
        z = jr.normal(key_, x_start.shape)
        
        sqrt_alpha_bar_t = jnp.sqrt(alpha_bars[t])
        sqrt_one_minus_alpha_bar_t = jnp.sqrt(1.0 - alpha_bars[t])
        
        x_t = sqrt_alpha_bar_t * x_start + sqrt_one_minus_alpha_bar_t * z
        
        chain_images.append(x_t)
        
    return chain_images

In [29]:
forward_timesteps = jnp.array([0, T//4, T//2, 3*T//4, T-1])
backward_timesteps = jnp.array([T-1, 3*T//4, T//2, T//4, 0])
print("Generating Forward Diffusion Chain (x_0 -> x_T)...")
forward_images = get_forward_chain(first_batch, forward_timesteps, key, batch_size=9)

for i, t in enumerate(forward_timesteps):
    create_image_grid(
        images=forward_images[i],
        num_rows=3, 
        num_cols=3, 
        filename=f"forward_T_{t}.png"
    )

Generating Forward Diffusion Chain (x_0 -> x_T)...
Saved: DDPM_result/forward_T_0.png
Saved: DDPM_result/forward_T_250.png
Saved: DDPM_result/forward_T_500.png
Saved: DDPM_result/forward_T_750.png
Saved: DDPM_result/forward_T_999.png


In [30]:
print("\nGenerating Backward Denoising Chain (x_T -> x_0)...")
key, key_ = jr.split(key)
all_samples_array = sample(final_state.apply_fn, final_state.params, key_, num_samples=9)


for t in backward_timesteps:

    array_index = T - t
    
    backward_images_at_t = all_samples_array[array_index]
    
    create_image_grid(
        images=backward_images_at_t,
        num_rows=3, 
        num_cols=3, 
        filename=f"backward_T_{t}.png"
    )


Generating Backward Denoising Chain (x_T -> x_0)...
Saved: DDPM_result/backward_T_999.png
Saved: DDPM_result/backward_T_750.png
Saved: DDPM_result/backward_T_500.png
Saved: DDPM_result/backward_T_250.png
Saved: DDPM_result/backward_T_0.png
